In [1]:
# ============================================================
# MTN NIGERIA PREPAID CHURN PREDICTION
# Phase 1 — Data Generation & Exploratory Data Analysis
# ============================================================
# Author: Elijjjaaaahhhhh
# Dataset: NGA_MTN_Subscribers_500K (synthetic)
# Seed: 42 (all random operations use this seed for reproducibility)
# ============================================================

# --- Core data libraries ---
import numpy as np
import pandas as pd

# --- Visualisation libraries ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Display settings ---
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.style.use('seaborn-v0_8-darkgrid')

# --- Reproducibility ---
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("All libraries imported successfully.")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print("Random seed set to 42 — all results are reproducible.")

All libraries imported successfully.
NumPy version: 2.4.3
Pandas version: 3.0.1
Random seed set to 42 — all results are reproducible.


In [2]:
# ============================================================
# CELL 2 — DATASET PARAMETERS
# Define all constants that control dataset generation.
# Change values here — nowhere else — to adjust the dataset.
# ============================================================

# --- Scale ---
N_SUBSCRIBERS = 500_000       # Total number of rows to generate
N_MONTHS      = 36            # Jan 2022 – Dec 2024

# --- Churn rate ---
BASE_MONTHLY_CHURN_RATE = 0.05   # 5% monthly churn (~25,000 churners)

# --- Nigerian states (36 states + FCT = 37 total) ---
NIGERIAN_STATES = [
    'Abia', 'Adamawa', 'Akwa Ibom', 'Anambra', 'Bauchi', 'Bayelsa',
    'Benue', 'Borno', 'Cross River', 'Delta', 'Ebonyi', 'Edo',
    'Ekiti', 'Enugu', 'Gombe', 'Imo', 'Jigawa', 'Kaduna', 'Kano',
    'Katsina', 'Kebbi', 'Kogi', 'Kwara', 'Lagos', 'Nasarawa', 'Niger',
    'Ogun', 'Ondo', 'Osun', 'Oyo', 'Plateau', 'Rivers', 'Sokoto',
    'Taraba', 'Yobe', 'Zamfara', 'FCT Abuja'
]

# --- States with higher churn (more competition, higher income) ---
HIGH_CHURN_STATES = ['Lagos', 'Kano', 'FCT Abuja', 'Rivers', 'Enugu']

# --- MTN Nigeria tariff plans ---
TARIFF_PLANS = ['MTNPulse', 'BetaGist', 'XtraValue', 'XtraSpecial', 'mPulse']

# --- Tariff plan weights (how common each plan is in the subscriber base) ---
# TalkMore is the most common prepaid plan
# SMEBundle is least common — it targets small businesses
TARIFF_WEIGHTS = [0.35, 0.15, 0.25, 0.15, 0.10]  # Must sum to 1.0

# --- State distribution weights ---
# Lagos, Kano, and Abuja have the largest subscriber bases
# We build this dynamically so every state gets a weight
STATE_WEIGHTS = []
for state in NIGERIAN_STATES:
    if state == 'Lagos':
        STATE_WEIGHTS.append(0.18)      # Lagos alone = 18% of subscribers
    elif state == 'Kano':
        STATE_WEIGHTS.append(0.07)      # Kano = 7%
    elif state == 'FCT Abuja':
        STATE_WEIGHTS.append(0.10)      # Abuja = 10%
    elif state == 'Rivers':
        STATE_WEIGHTS.append(0.06)      # Rivers = 6%
    elif state == 'Enugu':
        STATE_WEIGHTS.append(0.06)      # Enugu = 6%
    else:
        # Remaining 32 states share the remaining 53%
        STATE_WEIGHTS.append(0.53 / 32)

# Confirm weights sum to 1.0 (they must — this is a probability distribution)
assert round(sum(STATE_WEIGHTS), 6) == 1.0, "State weights must sum to 1.0"

# --- Observation months ---
import pandas as pd
OBSERVATION_MONTHS = pd.date_range(start='2022-01-01', periods=N_MONTHS, freq='MS')
# 'MS' = Month Start — generates the first day of each month

# --- Print confirmation ---
print(f"Dataset parameters defined.")
print(f"Total subscribers     : {N_SUBSCRIBERS:,}")
print(f"Observation period    : {OBSERVATION_MONTHS[0].strftime('%b %Y')} to {OBSERVATION_MONTHS[-1].strftime('%b %Y')}")
print(f"Number of months      : {N_MONTHS}")
print(f"Base monthly churn    : {BASE_MONTHLY_CHURN_RATE*100:.0f}%")
print(f"Number of states      : {len(NIGERIAN_STATES)}")
print(f"Tariff plans          : {TARIFF_PLANS}")
print(f"State weights sum     : {sum(STATE_WEIGHTS):.6f} ✓")


Dataset parameters defined.
Total subscribers     : 500,000
Observation period    : Jan 2022 to Dec 2024
Number of months      : 36
Base monthly churn    : 5%
Number of states      : 37
Tariff plans          : ['MTNPulse', 'BetaGist', 'XtraValue', 'XtraSpecial', 'mPulse']
State weights sum     : 1.000000 ✓
